# DE2 — Assignment 1: Streaming Pipeline
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

## 0. Setup

In [1]:
import os, sys, time, datetime, pathlib, json, csv
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType
)

DE2_SPARK_DRIVER_HOST  = os.environ.get("DE2_SPARK_DRIVER_HOST",  "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")


spark = (
    SparkSession.builder
    .appName("de2-assignment1")
    .master("local[*]")
    .config("spark.driver.host",        DE2_SPARK_DRIVER_HOST)
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS)
    .config("spark.ui.bindAddress",     DE2_SPARK_BIND_ADDRESS)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

show_spark_ui(spark)

Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## 0.1  Prepare Dataset as Streaming Source

**Track B – Smart Mobility IoT Stream.**  
We use `smart_mobility_dataset.csv` (5 000 rows, 5-minute intervals, March 2024, New York metro area)

The CSV is split into equal-sized JSON-lines batches written to `stream_source/`  
so the Structured Streaming file source picks them up incrementally.

In [2]:
import pandas as pd

SOURCE_DIR     = pathlib.Path("outputs/lab1/stream_source")
SINK_DIR       = pathlib.Path("outputs/lab1/stream_sink")
CHECKPOINT_DIR = pathlib.Path("outputs/lab1/checkpoint")
PROOF_DIR      = pathlib.Path("proof")

for d in [SOURCE_DIR, SINK_DIR, CHECKPOINT_DIR, PROOF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready:")
for d in [SOURCE_DIR, SINK_DIR, CHECKPOINT_DIR, PROOF_DIR]:
    print(" ", d)

CSV_PATH = pathlib.Path("smart_mobility_dataset.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at '{CSV_PATH}'. "
        "Place smart_mobility_dataset.csv next to this notebook."
    )

df_full = pd.read_csv(CSV_PATH)
df_full = df_full.rename(columns={"Road_Occupancy_%": "Road_Occupancy_pct"})
print(f"Dataset loaded: {df_full.shape[0]} rows × {df_full.shape[1]} columns")
print("Columns:", list(df_full.columns))
print("Timestamp range:", df_full["Timestamp"].iloc[0], "->", df_full["Timestamp"].iloc[-1])

BATCH_SIZE   = 500
INIT_BATCHES = 8

batches = [df_full.iloc[i : i + BATCH_SIZE] for i in range(0, len(df_full), BATCH_SIZE)]
print(f"\nSplit into {len(batches)} batches of ~{BATCH_SIZE} rows each.")

print("\nWriting initial data batches ...")
for idx, batch in enumerate(batches[:INIT_BATCHES]):
    path = SOURCE_DIR / f"batch_{idx:03d}.json"
    batch.to_json(path, orient="records", lines=True)
    print(f"  wrote {path.name}  ({len(batch)} rows)")

REMAINING_BATCHES = batches[INIT_BATCHES:]
print(f"Done. {len(REMAINING_BATCHES)} batch(es) reserved for the optimised run.")

Directories ready:
  outputs\lab1\stream_source
  outputs\lab1\stream_sink
  outputs\lab1\checkpoint
  proof
Dataset loaded: 5000 rows × 15 columns
Columns: ['Timestamp', 'Latitude', 'Longitude', 'Vehicle_Count', 'Traffic_Speed_kmh', 'Road_Occupancy_pct', 'Traffic_Light_State', 'Weather_Condition', 'Accident_Report', 'Sentiment_Score', 'Ride_Sharing_Demand', 'Parking_Availability', 'Emission_Levels_g_km', 'Energy_Consumption_L_h', 'Traffic_Condition']
Timestamp range: 2024-03-01 00:00:00 -> 2024-03-18 08:35:00

Split into 10 batches of ~500 rows each.

Writing initial data batches ...
  wrote batch_000.json  (500 rows)
  wrote batch_001.json  (500 rows)
  wrote batch_002.json  (500 rows)
  wrote batch_003.json  (500 rows)
  wrote batch_004.json  (500 rows)
  wrote batch_005.json  (500 rows)
  wrote batch_006.json  (500 rows)
  wrote batch_007.json  (500 rows)
Done. 2 batch(es) reserved for the optimised run.


## 1. Define Your Schema and Stream Source

**Track B - Smart Mobility IoT Stream.**

| Field | Type | Notes |
|---|---|---|
| `Timestamp` | TimestampType | sensor wall-clock time (UTC) - event-time column |
| `Latitude` | DoubleType | GPS latitude |
| `Longitude` | DoubleType | GPS longitude |
| `Vehicle_Count` | IntegerType | vehicles detected in the interval |
| `Traffic_Speed_kmh` | DoubleType | average speed (km/h) |
| `Road_Occupancy_pct` | DoubleType | road occupancy (%) |
| `Traffic_Light_State` | StringType | Green / Yellow / Red |
| `Weather_Condition` | StringType | Clear / Rain / Fog / Snow |
| `Accident_Report` | IntegerType | 1 = accident reported, 0 = none |
| `Sentiment_Score` | DoubleType | social-media sentiment (-1 to +1) |
| `Ride_Sharing_Demand` | IntegerType | ride-share requests in interval |
| `Parking_Availability` | IntegerType | free parking spots |
| `Emission_Levels_g_km` | DoubleType | CO₂ equivalent (g/km) |
| `Energy_Consumption_L_h` | DoubleType | fuel consumption (L/h) |
| `Traffic_Condition` | StringType | Low / High - used as grouping key |

In [3]:
mobility_schema = StructType([
    StructField("Timestamp",              TimestampType(), nullable=False),
    StructField("Latitude",               DoubleType(),    nullable=True),
    StructField("Longitude",              DoubleType(),    nullable=True),
    StructField("Vehicle_Count",          IntegerType(),   nullable=True),
    StructField("Traffic_Speed_kmh",      DoubleType(),    nullable=True),
    StructField("Road_Occupancy_pct",     DoubleType(),    nullable=True),
    StructField("Traffic_Light_State",    StringType(),    nullable=True),
    StructField("Weather_Condition",      StringType(),    nullable=True),
    StructField("Accident_Report",        IntegerType(),   nullable=True),
    StructField("Sentiment_Score",        DoubleType(),    nullable=True),
    StructField("Ride_Sharing_Demand",    IntegerType(),   nullable=True),
    StructField("Parking_Availability",   IntegerType(),   nullable=True),
    StructField("Emission_Levels_g_km",   DoubleType(),    nullable=True),
    StructField("Energy_Consumption_L_h", DoubleType(),    nullable=True),
    StructField("Traffic_Condition",      StringType(),    nullable=True),
])

EVENT_TIME_COL  = "Timestamp"
WINDOW_DURATION = "1 minute"
WATERMARK_DELAY = "30 seconds"

raw_stream = (
    spark.readStream
         .format("json")
         .schema(mobility_schema)
         .option("maxFilesPerTrigger", 2)
         .load(str(SOURCE_DIR))
)

print("Stream schema:")
raw_stream.printSchema()
print(f"isStreaming = {raw_stream.isStreaming}")

Stream schema:
root
 |-- Timestamp: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Vehicle_Count: integer (nullable = true)
 |-- Traffic_Speed_kmh: double (nullable = true)
 |-- Road_Occupancy_pct: double (nullable = true)
 |-- Traffic_Light_State: string (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Accident_Report: integer (nullable = true)
 |-- Sentiment_Score: double (nullable = true)
 |-- Ride_Sharing_Demand: integer (nullable = true)
 |-- Parking_Availability: integer (nullable = true)
 |-- Emission_Levels_g_km: double (nullable = true)
 |-- Energy_Consumption_L_h: double (nullable = true)
 |-- Traffic_Condition: string (nullable = true)

isStreaming = True


## 2. Windowed Aggregation + Watermark

In [4]:
agg_stream = (
    raw_stream
    .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY)
    .groupBy(
        F.window(F.col(EVENT_TIME_COL), WINDOW_DURATION),
        F.col("Traffic_Condition")
    )
    .agg(
        F.count("*").alias("sensor_readings"),
        F.avg("Vehicle_Count").alias("avg_vehicle_count"),
        F.avg("Traffic_Speed_kmh").alias("avg_speed_kmh"),
        F.sum("Accident_Report").alias("total_accidents"),
        F.avg("Emission_Levels_g_km").alias("avg_emission_g_km"),
        F.avg("Road_Occupancy_pct").alias("avg_road_occupancy_pct"),
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end",   F.col("window.end"))
    .drop("window")
)

## 3. Write Stream to Parquet

Key choices:
| Option | Value | Rationale |
|---|---|---|
| `outputMode` | `append` | Required for windowed aggregations with watermark - only finalised windows are emitted |
| `format` | `parquet` | Columnar, compressed, splittable - ideal for downstream batch analytics |
| `trigger` | `processingTime 10 seconds` | Controls micro-batch cadence (baseline) |
| `checkpointLocation` | `outputs/lab1/checkpoint` | Ensures exactly-once semantics across restarts |

In [5]:
TRIGGER_INTERVAL = "10 seconds"

query = (
    agg_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", str(SINK_DIR))
    .option("checkpointLocation", str(CHECKPOINT_DIR))
    .partitionBy("Traffic_Condition")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .queryName("mobility_window_agg")
    .start()
)

print(f"Query '{query.name}' started.")
print(f"Query ID : {query.id}")
print(f"Run  ID  : {query.runId}")
print(f"Status   : {query.status}")

Query 'mobility_window_agg' started.
Query ID : a4d1b572-2288-489c-ad86-46200aad9a03
Run  ID  : 66fd30da-a0d0-4817-92d2-6736333cfd65
Status   : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## 4. Monitor and Capture Evidence

In [6]:
TARGET_BATCHES = 5
POLL_INTERVAL  = 5
MAX_WAIT       = 180

print(f"Waiting for {TARGET_BATCHES} micro-batches "
      f"(max {MAX_WAIT}s, polling every {POLL_INTERVAL}s) ...")

elapsed = 0
while elapsed < MAX_WAIT:
    prog     = query.lastProgress
    batch_id = prog.get("batchId", -1) if prog else -1
    print(f"  [{elapsed:>4}s] batchId={batch_id}  "
          f"inputRows={prog.get('numInputRows', 0) if prog else 0}")
    if batch_id >= TARGET_BATCHES - 1:
        print(f"  -> Reached {TARGET_BATCHES} batches.")
        break
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

if elapsed >= MAX_WAIT:
    print("WARNING: timeout reached before accumulating enough batches.")

print("\n lastProgress")
print(json.dumps(query.lastProgress, indent=2, default=str))

Waiting for 5 micro-batches (max 180s, polling every 5s) ...
  [   0s] batchId=-1  inputRows=0
  [   5s] batchId=-1  inputRows=0
  [  10s] batchId=-1  inputRows=0
  [  15s] batchId=-1  inputRows=0
  [  20s] batchId=-1  inputRows=0
  [  25s] batchId=-1  inputRows=0
  [  30s] batchId=-1  inputRows=0
  [  35s] batchId=-1  inputRows=0
  [  40s] batchId=-1  inputRows=0
  [  45s] batchId=-1  inputRows=0
  [  50s] batchId=-1  inputRows=0
  [  55s] batchId=-1  inputRows=0
  [  60s] batchId=-1  inputRows=0
  [  65s] batchId=-1  inputRows=0
  [  70s] batchId=-1  inputRows=0
  [  75s] batchId=-1  inputRows=0
  [  80s] batchId=-1  inputRows=0
  [  85s] batchId=-1  inputRows=0
  [  90s] batchId=-1  inputRows=0
  [  95s] batchId=-1  inputRows=0
  [ 100s] batchId=-1  inputRows=0
  [ 105s] batchId=-1  inputRows=0
  [ 110s] batchId=-1  inputRows=0
  [ 115s] batchId=-1  inputRows=0
  [ 120s] batchId=-1  inputRows=0
  [ 125s] batchId=-1  inputRows=0
  [ 130s] batchId=-1  inputRows=0
  [ 135s] batchId=-1 

In [7]:
plan_path = PROOF_DIR / "plan_streaming.txt"

plan_df = (
    spark.read
         .schema(mobility_schema)
         .json(str(SOURCE_DIR))
         .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY)
         .groupBy(
             F.window(F.col(EVENT_TIME_COL), WINDOW_DURATION),
             F.col("Traffic_Condition")
         )
         .agg(
             F.count("*").alias("sensor_readings"),
             F.avg("Vehicle_Count").alias("avg_vehicle_count"),
             F.avg("Traffic_Speed_kmh").alias("avg_speed_kmh"),
             F.sum("Accident_Report").alias("total_accidents"),
             F.avg("Emission_Levels_g_km").alias("avg_emission_g_km"),
             F.avg("Road_Occupancy_pct").alias("avg_road_occupancy_pct"),
         )
)

plan_lines = [
    "=" * 70,
    "STREAMING PIPELINE - EXECUTION PLAN  (Track B · Smart Mobility)",
    f"Generated : {datetime.datetime.utcnow().isoformat()} UTC",
    "=" * 70,
    "",
    "Logical Plan (unoptimised)",
    plan_df._jdf.queryExecution().logical().toString(),
    "",
    "Analysed Logical Plan",
    plan_df._jdf.queryExecution().analyzed().toString(),
    "",
    "Optimised Logical Plan",
    plan_df._jdf.queryExecution().optimizedPlan().toString(),
    "",
    "Physical Plan",
    plan_df._jdf.queryExecution().executedPlan().toString(),
]

plan_path.write_text("\n".join(plan_lines), encoding="utf-8")
print(f"Plan written -> {plan_path}")

ui_url = spark.sparkContext.uiWebUrl or "http://localhost:4040"
print(f"\nScreenshot the Streaming UI now:")
print(f"   {ui_url}/StreamingQuery/")
print("   (capture 'Input Rate' and 'Processing Rate' charts)")

Plan written -> proof\plan_streaming.txt

Screenshot the Streaming UI now:
   http://127.0.0.1:4040/StreamingQuery/
   (capture 'Input Rate' and 'Processing Rate' charts)


In [8]:
parquet_files = list(SINK_DIR.rglob("*.parquet"))
print(f"Parquet part-files found: {len(parquet_files)}")
for p in parquet_files[:5]:
    print(" ", p)

if parquet_files:
    result_df = spark.read.parquet(str(SINK_DIR))
    print(f"\nTotal rows in sink: {result_df.count()}")
    result_df.show(10, truncate=False)
else:
    print("No Parquet files yet - ensure the query has processed at least "
          "one complete window before reading.")

Parquet part-files found: 4
  outputs\lab1\stream_sink\action=add_to_cart\part-00002-6601bd02-c72c-4f87-a4e8-2fb16068d6be.c000.snappy.parquet
  outputs\lab1\stream_sink\action=purchase\part-00003-c4991b65-484e-4df2-af1d-92d3149afd48.c000.snappy.parquet
  outputs\lab1\stream_sink\action=view\part-00001-59e991e6-8db0-4477-92a5-23e90ae4e663.c000.snappy.parquet
  outputs\lab1\stream_sink\action=wishlist\part-00000-00dcb953-b0df-4e77-8c2f-568aabc5dd74.c000.snappy.parquet

Total rows in sink: 4
+-----------+------------------+------------------+------------+-------------------+-------------------+-----------+
|event_count|total_amount      |avg_amount        |unique_users|window_start       |window_end         |action     |
+-----------+------------------+------------------+------------+-------------------+-------------------+-----------+
|3          |508.39000000000004|169.46333333333334|3           |2026-05-11 14:30:00|2026-05-11 14:31:00|add_to_cart|
|3          |1018.18           |339.39

## 5. Optimize and Re-Measure

**Baseline configuration**

| Parameter | Baseline |
|---|---|
| `trigger` | `10 seconds` |
| `maxFilesPerTrigger` | 2 |
| `spark.sql.shuffle.partitions` | 4 |
| `WINDOW_DURATION` | `1 minute` |
| `WATERMARK_DELAY` | `30 seconds` |

In [9]:
prog_baseline   = query.lastProgress or {}
baseline_metrics = {
    "config":                "baseline",
    "trigger_interval":      "10s",
    "max_files_per_trigger": 2,
    "shuffle_partitions":    4,
    "batch_id":              prog_baseline.get("batchId", ""),
    "num_input_rows":        prog_baseline.get("numInputRows", ""),
    "input_rows_per_sec":    prog_baseline.get("inputRowsPerSecond", ""),
    "proc_rows_per_sec":     prog_baseline.get("processedRowsPerSecond", ""),
    "trigger_exec_ms":       (prog_baseline.get("durationMs") or {}).get("triggerExecution", ""),
    "batch_duration_ms":     (prog_baseline.get("durationMs") or {}).get("getBatch", ""),
}
print("Baseline metrics captured:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v}")

query.stop()
print("\nBaseline query stopped.")

Baseline metrics captured:
  config: baseline
  trigger_interval: 10s
  max_files_per_trigger: 2
  shuffle_partitions: 4
  batch_id: 
  num_input_rows: 
  input_rows_per_sec: 
  proc_rows_per_sec: 
  trigger_exec_ms: 
  batch_duration_ms: 

Baseline query stopped.


In [10]:
print("Writing additional batches for optimised run ...")
for idx, batch in enumerate(REMAINING_BATCHES, start=INIT_BATCHES):
    path = SOURCE_DIR / f"batch_{idx:03d}.json"
    batch.to_json(path, orient="records", lines=True)
    print(f"  wrote {path.name}  ({len(batch)} rows)")
print("Done.")

spark.conf.set("spark.sql.shuffle.partitions", "8")

OPTIMIZED_CHECKPOINT_DIR = pathlib.Path("outputs/lab1/checkpoint_opt")
OPTIMIZED_SINK_DIR       = pathlib.Path("outputs/lab1/stream_sink_opt")
OPTIMIZED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OPTIMIZED_SINK_DIR.mkdir(parents=True, exist_ok=True)

raw_stream_opt = (
    spark.readStream
         .format("json")
         .schema(mobility_schema)
         .option("maxFilesPerTrigger", 4)
         .load(str(SOURCE_DIR))
)

agg_stream_opt = (
    raw_stream_opt
    .withWatermark(EVENT_TIME_COL, "30 seconds")
    .groupBy(
        F.window(F.col(EVENT_TIME_COL), WINDOW_DURATION),
        F.col("Traffic_Condition")
    )
    .agg(
        F.count("*").alias("sensor_readings"),
        F.avg("Vehicle_Count").alias("avg_vehicle_count"),
        F.avg("Traffic_Speed_kmh").alias("avg_speed_kmh"),
        F.sum("Accident_Report").alias("total_accidents"),
        F.avg("Emission_Levels_g_km").alias("avg_emission_g_km"),
        F.avg("Road_Occupancy_pct").alias("avg_road_occupancy_pct"),
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end",   F.col("window.end"))
    .drop("window")
)

query_opt = (
    agg_stream_opt.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", str(OPTIMIZED_SINK_DIR))
    .option("checkpointLocation", str(OPTIMIZED_CHECKPOINT_DIR))
    .partitionBy("Traffic_Condition")
    .trigger(processingTime="5 seconds")
    .queryName("mobility_window_agg_opt")
    .start()
)

print(f"Optimised query '{query_opt.name}' started.")

elapsed = 0
while elapsed < MAX_WAIT:
    prog     = query_opt.lastProgress
    batch_id = prog.get("batchId", -1) if prog else -1
    print(f"  [{elapsed:>4}s] batchId={batch_id}  "
          f"inputRows={prog.get('numInputRows', 0) if prog else 0}")
    if batch_id >= TARGET_BATCHES - 1:
        print(f"  -> Reached {TARGET_BATCHES} batches (optimised run).")
        break
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

print("\n lastProgress (optimised)")
print(json.dumps(query_opt.lastProgress, indent=2, default=str))

Writing additional batches for optimised run ...
  wrote batch_008.json  (500 rows)
  wrote batch_009.json  (500 rows)
Done.
Optimised query 'mobility_window_agg_opt' started.
  [   0s] batchId=-1  inputRows=0
  [   5s] batchId=4  inputRows=0
  -> Reached 5 batches (optimised run).

 lastProgress (optimised)
{
  "id": "731a44aa-6b6d-4aec-85fb-2cd814fda978",
  "runId": "91660e62-0d95-453e-a72c-6f0f59d12460",
  "name": "mobility_window_agg_opt",
  "timestamp": "2026-06-01T13:14:35.598Z",
  "batchId": 4,
  "batchDuration": 19,
  "durationMs": {
    "triggerExecution": 19,
    "latestOffset": 9
  },
  "eventTime": {},
  "stateOperators": [],
  "sources": [
    {
      "description": "FileStreamSource[file:/c:/Users/Enzor/OneDrive/Bureau/ESIEE/E4_Data_Eng_2/Lab1/outputs/lab1/stream_source]",
      "startOffset": "{\"logOffset\":3}",
      "endOffset": "{\"logOffset\":3}",
      "latestOffset": "None",
      "numInputRows": 0,
      "inputRowsPerSecond": 0.0,
      "processedRowsPerSecond": 

In [11]:
prog_opt = query_opt.lastProgress or {}
optimized_metrics = {
    "config":                "optimized",
    "trigger_interval":      "5s",
    "max_files_per_trigger": 4,
    "shuffle_partitions":    8,
    "batch_id":              prog_opt.get("batchId", ""),
    "num_input_rows":        prog_opt.get("numInputRows", ""),
    "input_rows_per_sec":    prog_opt.get("inputRowsPerSecond", ""),
    "proc_rows_per_sec":     prog_opt.get("processedRowsPerSecond", ""),
    "trigger_exec_ms":       (prog_opt.get("durationMs") or {}).get("triggerExecution", ""),
    "batch_duration_ms":     (prog_opt.get("durationMs") or {}).get("getBatch", ""),
}
print("Optimised metrics captured:")
for k, v in optimized_metrics.items():
    print(f"  {k}: {v}")

query_opt.stop()
print("\nOptimised query stopped.")

Optimised metrics captured:
  config: optimized
  trigger_interval: 5s
  max_files_per_trigger: 4
  shuffle_partitions: 8
  batch_id: 4
  num_input_rows: 0
  input_rows_per_sec: 0.0
  proc_rows_per_sec: 0.0
  trigger_exec_ms: 19
  batch_duration_ms: 

Optimised query stopped.


## 6. Fill Metrics Log

Write `lab1_metrics_log.csv` with one row per configuration (baseline + optimised).

In [12]:
METRICS_CSV = pathlib.Path("lab1_metrics_log.csv")

fieldnames = [
    "config", "trigger_interval", "max_files_per_trigger",
    "shuffle_partitions", "batch_id", "num_input_rows",
    "input_rows_per_sec", "proc_rows_per_sec",
    "trigger_exec_ms", "batch_duration_ms"
]

rows = [baseline_metrics, optimized_metrics]

with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Metrics written -> {METRICS_CSV}")

print("\nContents of lab1_metrics_log.csv:")
print(",".join(fieldnames))
for row in rows:
    print(",".join(str(row[f]) for f in fieldnames))

print("\nThroughput comparison")
b_tput = baseline_metrics.get("proc_rows_per_sec") or 0
o_tput = optimized_metrics.get("proc_rows_per_sec") or 0
try:
    delta = (float(o_tput) - float(b_tput)) / (float(b_tput) + 1e-9) * 100
    print(f"  Baseline  processedRowsPerSecond : {b_tput}")
    print(f"  Optimised processedRowsPerSecond : {o_tput}")
    print(f"  Change : {delta:+.1f}%")
except (TypeError, ValueError):
    print("  (metrics not yet available - re-run after queries complete)")

Metrics written -> lab1_metrics_log.csv

Contents of lab1_metrics_log.csv:
config,trigger_interval,max_files_per_trigger,shuffle_partitions,batch_id,num_input_rows,input_rows_per_sec,proc_rows_per_sec,trigger_exec_ms,batch_duration_ms
baseline,10s,2,4,,,,,,
optimized,5s,4,8,4,0,0.0,0.0,19,

Throughput comparison
  Baseline  processedRowsPerSecond : 0
  Optimised processedRowsPerSecond : 0
  Change : +0.0%


## 7. Cleanup

In [13]:
print(" Output directory summary")
for d in [SINK_DIR, OPTIMIZED_SINK_DIR, CHECKPOINT_DIR,
          OPTIMIZED_CHECKPOINT_DIR, PROOF_DIR]:
    files = list(pathlib.Path(d).rglob("*")) if pathlib.Path(d).exists() else []
    print(f"  {d}: {len(files)} file(s)")

print(f"  {METRICS_CSV}: {'exists' if METRICS_CSV.exists() else 'MISSING'}")
print(f"  {PROOF_DIR / 'plan_streaming.txt'}: "
      f"{'exists' if (PROOF_DIR / 'plan_streaming.txt').exists() else 'MISSING'}")

spark.stop()
print("\nDone.")

 Output directory summary
  outputs\lab1\stream_sink: 23 file(s)
  outputs\lab1\stream_sink_opt: 29 file(s)
  outputs\lab1\checkpoint: 92 file(s)
  outputs\lab1\checkpoint_opt: 110 file(s)
  proof: 1 file(s)
  lab1_metrics_log.csv: exists
  proof\plan_streaming.txt: exists

Done.
